In [9]:
from openadmet.toolkit.database.chembl import LogPDCurator
from openadmet.toolkit.chemoinformatics.rdkit_funcs import canonical_smiles, smiles_to_inchikey
from tqdm.auto import tqdm
tqdm.pandas()
import datamol as dm
import os
import subprocess
import numpy as np

# Curating LogD data from ChEMBL and pushing to a remote intake catalog

Our goal is to curate activity data from ChEMBL and push this to a remote location with a catalog that can be used by others to look up our data. This will enable consistency and rapid dissemination of our work as well as an over-time evolution of our data curation practices. 

We use the `Intake` package for a lightweight self-describing data parsing workflow. Read more about intake here: https://intake.readthedocs.io/en/latest/index.html


Here we gather `LogD` data permissivley from ChEMBL (ie without activity based curation). LogD has BAO BAO_0000100 (small molecule physprop format) and no target

We then aggregate  measurements on the same compound by taking the mean and median. This is the most basic form of curation available, but serves as a good baseline for our initial models. 



## gather ChEMBL data

First we need to gather in our data from ChEMBL using our SQL API defined in `openadmet-toolkit`

We use `OPENADMET_CANONICAL_SMILES` and `OPENADMET_INCHIKEY` to distinguish our ML ready representation from the source SMILES

In [5]:
def gather_chembl_data_logD(chembl_ver: int):
    print(f"working on target")
    pctc = LogPDCurator( version=chembl_ver, standard_type="LogD")
    activity_data = pctc.get_activity_data(return_as="df")
    print("canonicalising raw data")
    with dm.without_rdkit_log():
        activity_data["OPENADMET_CANONICAL_SMILES"] = activity_data["canonical_smiles"].progress_apply(lambda x: canonical_smiles(x))
        activity_data["OPENADMET_INCHIKEY"] = activity_data["OPENADMET_CANONICAL_SMILES"].progress_apply(lambda x: smiles_to_inchikey(x))
    # important to canonicalise here so compound deduplication is done correctly
    aggregated_activity = pctc.aggregate_activity_data_by_compound(canonicalise=True)
    print("smiles duplicates", aggregated_activity["OPENADMET_CANONICAL_SMILES"].duplicated().sum())
    print("inchikey duplicates", aggregated_activity["OPENADMET_INCHIKEY"].duplicated().sum())
    return aggregated_activity, activity_data
        

In [6]:
chembl_ver = 35

In [7]:
from openadmet.toolkit.webservices.credentials import S3Settings
from openadmet.toolkit.webservices.s3 import S3Bucket

# Setup S3

After curating our data we would like to push to a remote bucket to save both the raw data and the catalog

In [ ]:
os.environ["AWS_ACCESS_KEY_ID"] = subprocess.check_output(
    ["aws", "configure", "get", "aws_access_key_id"],
    text=True
).strip()

os.environ["AWS_SECRET_ACCESS_KEY"] = subprocess.check_output(
    ["aws", "configure", "get", "aws_secret_access_key"],
    text=True
).strip()

In [13]:
from pathlib import Path
import configparser
cred_path = Path.home() / ".aws" / "credentials"
if cred_path.exists():
    config = configparser.ConfigParser()
    config.read(cred_path)

    os.environ["AWS_ACCESS_KEY_ID"] = config.get("default", "aws_access_key_id")
    os.environ["AWS_SECRET_ACCESS_KEY"] = config.get(
        "default", "aws_secret_access_key"
    )
    print("✅ AWS Credentials injected into OS environment.")

✅ AWS Credentials injected into OS environment.


In [14]:
settings = S3Settings()

In [15]:
bucket = "openadmet-data-public-dev"

In [16]:
bucket = S3Bucket.from_settings(settings, bucket)

In [17]:
import datetime

In [18]:
t = datetime.datetime.now()

In [19]:
date = t.strftime("%Y-%m-%d")

In [20]:
location=f"ChEMBL{chembl_ver}_LogD"

In [21]:
import os
from pathlib import Path

location_path = Path(location)

In [22]:
location_path.mkdir(exist_ok=True)

We can just use a one off here as no need to loop over targets e.g in activity curation

In [28]:
uris_raw = {}
uris_agg = {}
target = "LogD"
agg, raw  = gather_chembl_data_logD(chembl_ver)
fname_agg = f"ChEMBL_LogD_{target}_aggregated.parquet"
fname_raw = f"ChEMBL_LogD_{target}_raw.parquet"
    
agg.reset_index(drop=True).to_parquet(location_path/fname_agg, index=False)
raw.reset_index(drop=True).to_parquet(location_path/fname_raw, index=False)
    
bucket_destination_agg = location + "/" + fname_agg
bucket.push_file(location_path/fname_agg, bucket_destination_agg)
bucket_destination_raw = location + "/" + fname_raw
bucket.push_file(location_path/fname_raw, bucket_destination_raw)

# get S3 URIs
uri_agg = bucket.to_uri(bucket_destination_agg)
uris_agg[target] = uri_agg

uri_raw = bucket.to_uri(bucket_destination_raw)
uris_raw[target] = uri_raw

    


working on target


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

canonicalising raw data


  0%|          | 0/26703 [00:00<?, ?it/s]

  0%|          | 0/26703 [00:00<?, ?it/s]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  0%|          | 0/26703 [00:00<?, ?it/s]

  0%|          | 0/26703 [00:00<?, ?it/s]

smiles duplicates 0
inchikey duplicates 0


# Build the Intake Catalog

We have sucessfully aggregted our data and pushed it to a remote destination. Now for others to consume our data, we are going to make an `Intake` catalog such that our data can be readily made available. 

The workflow here is drawn from the `creator` walkthrough from the main intake tutorials https://intake.readthedocs.io/en/latest/walkthrough2.html

TODO: add descriptions to the catalog

In [29]:
import intake

In [30]:
intake.Catalog?

Init signature:
intake.Catalog(
    entries: 'Iterable[ReaderDescription] | Mapping | None' = None,
    aliases: 'dict[str, int] | None' = None,
    data: 'Iterable[DataDescription] | Mapping' = None,
    user_parameters: 'dict[str, BaseUserParameter] | None' = None,
    parameter_overrides: 'dict[str, Any] | None' = None,
    metadata: 'dict | None' = None,
)
Docstring:      A collection of data and reader descriptions.
File:           ~/miniforge3/envs/admet-env/lib/python3.11/site-packages/intake/readers/entry.py
Type:           type
Subclasses:     THREDDSCatalog

In [31]:
cat = intake.entry.Catalog()

In [32]:
uris_agg

{'LogD': 's3://openadmet-data-public-dev/ChEMBL35_LogD/ChEMBL_LogD_LogD_aggregated.parquet'}

In [33]:
uris_raw

{'LogD': 's3://openadmet-data-public-dev/ChEMBL35_LogD/ChEMBL_LogD_LogD_raw.parquet'}

In [34]:
for k,v in uris_agg.items():
    cat[k+"_aggregated"] = intake.readers.PandasParquet(v)

In [35]:
for k,v in uris_raw.items():
    cat[k+"_raw"] = intake.readers.PandasParquet(v)

## Push the Catalog

Ok now we have made the catalog, lets push it to the remote location so it can live alongside the data. 

The catalog can then be used from S3 or from github etc, anything that exposes a file-like API. 

In [ ]:
cat

In [ ]:
catname = f"CATALOG_{location}.yaml"

In [ ]:
cat.to_yaml_file(catname)

In [ ]:
cat_location = location+ "/" +catname

In [ ]:
cat_location

In [ ]:
bucket.push_file(catname, cat_location)

In [ ]:
cat_uri = bucket.to_uri(cat_location)

In [ ]:
# Now can read the catalog from URI
# cat = intake.Catalog.from_yaml_file("s3://openadmet-data-public-dev/ChEMBL34_permissive_2025-02-12/CATALOG_ChEMBL34_permissive_2025-02-12.yaml")

## Trimming data
Optional code to trim outliers to further curate the data

In [36]:
def trim_outliers_to_nan(
    df,
    column,
    lower=None,
    upper=None,
    lower_quantile=0.01,
    upper_quantile=0.99,
):
    mask = df[column].notna()

    if lower is None:
        lower = df.loc[mask, column].quantile(lower_quantile)
    if upper is None:
        upper = df.loc[mask, column].quantile(upper_quantile)

    trim_mask = mask & ((df[column] < lower) | (df[column] > upper))
    df_trimmed = df.copy()
    df_trimmed.loc[trim_mask, column] = np.nan

    summary = {
        'lower': lower,
        'upper': upper,
        'trimmed': int(trim_mask.sum()),
        'remaining': int(df_trimmed[column].notna().sum()),
    }

    return df_trimmed, summary

In [38]:
target_agg = cat["LogD_aggregated"].read()
df_trimmed, summary = trim_outliers_to_nan(
    target_agg,
    column="standard_value_mean",
    lower=-4,
    upper=8,
 )

print(
    f"Trimmed {summary['trimmed']} LogD values outside [{summary['lower']}, {summary['upper']}] "
    f"to NaN. Total rows unchanged: {len(target_agg)}. "
    f"Remaining labeled values: {summary['remaining']}"
 )

Trimmed 223 LogD values outside [-4, 8] to NaN. Total rows unchanged: 23896. Remaining labeled values: 22806


In [ ]:
uris_trimmed = {}

fname_trimmed = f"trimmed/ChEMBL_LogD_{target}_aggregated.parquet"
Path(location_path/fname_trimmed).parent.mkdir(exist_ok=True, parents=True)

df_trimmed.reset_index(drop=True).to_parquet(location_path/fname_trimmed, index=False)

bucket_destination_trimmed = location + "/" + fname_trimmed
bucket.push_file(location_path/fname_trimmed, bucket_destination_trimmed)

# get S3 URIs
uri_trimmed = bucket.to_uri(bucket_destination_trimmed)
uris_trimmed[target] = uri_trimmed

In [ ]:
for k, v in uris_trimmed.items():
    cat[k + "_trimmed_aggregated"] = intake.readers.PandasParquet(v)

# Re-push updated catalog with trimmed entries
cat.to_yaml_file(catname)
bucket.push_file(catname, cat_location)